# Imports

## Bibliothèques

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

## Dataset

In [ ]:
df = pd.read_csv('../data/df_complet.csv', sep=';', dtype={'Code_INSEE': str})
print(f"Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")

print(f"\nDistribution de la cible (vote politique) :")
print(df['Résultat'].value_counts())

print('\nColonnes :')
print(df.columns)

df = df.dropna()
print(f"Valeurs manquantes supprimées, df final : {df.shape[0]} lignes, {df.shape[1]} colonnes")

## Feature engineering

In [ ]:
df = df.drop(columns=['Année', 'Code_INSEE'])


def convert_columns_to_percentages(df, list_columns, divider_column):
    for col in list_columns:
        df[col] = df[col] / df[divider_column] * 100
    return df


# Transformation des genres, catégories socio-professionnelles, tranches d'âge et statuts maritaux en pourcentages de la population active
columns_to_convert = [
    'Hommes', 
    'Femmes', 
    'Agriculteurs', 
    'Artisans', 
    'Cadres', 
    'Intermédiaires', 
    'Employés', 
    'Ouvriers',
    'Retraités',
    'Etudiants',
    'Inactifs',
    '15-24 ans',
    '25-39 ans',
    '40-54 ans',
    '55-64 ans',
    '65-79 ans',
    '80 ans et +',
    'Mariés',
    'Pacsés',
    'Concubinage',
    'Veufs',
    'Divorcés',
    'Célibataires',
]

df = convert_columns_to_percentages(df, columns_to_convert, 'Population_active')



# Transformation des compositions de ménages en pourcentage de la population avec enfants

columns_to_convert_household = [
    'Personne seule',
    'Homme seul',
    'Femme seule',
    'Colocation',
    'Famille',
    'Famille monoparentale',
    'Couple sans enfant',
    'Couple avec enfants',
]
df = convert_columns_to_percentages(df, columns_to_convert_household, 'Population avec enfants')

display(df)


# Random Forest

## Split test-train

In [ ]:
X = df.drop(columns=['Résultat'])
y = df['Résultat']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print("✓ Données préparées")
print(f"  Train: {X_train.shape}")
print(f"  Test: {X_test.shape}")

## Fonctions

In [ ]:
groups = ['centre', 'droite', 'gauche']


def train_model_and_detect_overfitting(model):
    model.fit(X_train, y_train)

    y_pred_test = model.predict(X_test)
    y_pred_train = model.predict(X_train)


    train_score = model.score(X_train, y_train)
    test_score = model.score(X_test, y_test)
    gap = abs(train_score - test_score)


    print(f"  Train : {train_score:.2%}")
    print(f"  Test  : {test_score:.2%}")
    print(f"  Gap   : {gap:.2%}\n")


    if gap > 0.10:
        print("⚠️ OVERFITTING détecté !")
    elif train_score < 0.75:
        print("⚠️ UNDERFITTING détecté !")
    else:
        print("✅ Bon équilibre\n")
    
    return y_pred_test, y_pred_train


def display_confusion_matrix(y, y_pred):
    matrix = confusion_matrix(y, y_pred, labels=groups)

    plt.figure(figsize=(8, 6))
    sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=[f'Prédit {g}' for g in groups],
                yticklabels=[f'Réel {g}' for g in groups])
    plt.title('Matrice de confusion', fontsize=14, pad=15)
    plt.tight_layout()
    plt.show()


def display_metrics(y, y_pred):
    print("\n\nRapport de classification :")
    print(classification_report(y, y_pred, target_names=groups))

## Random Forest weighted

### Recherche des meilleurs hyperparamètres - Random Search

In [ ]:
# Modèle avec poids de classes ajustés
forest_weighted = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(random_state=42, max_depth=10, class_weight='balanced'))
])


param_distributions = {
    'classifier__n_estimators': [200, 300],
    'classifier__min_samples_split': [12, 15],
    'classifier__min_samples_leaf': [1, 3, 5],
    'classifier__max_features': ['sqrt', 'log2']
}


# Random Search
print("Lancement de Random Search...")
random_search = RandomizedSearchCV(
    forest_weighted,
    param_distributions,
    n_iter=10,  # Teste 30 combinaisons aléatoires
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    random_state=42,
    verbose=1
)


# Recherche
random_search.fit(X_train, y_train)

print("\n✓ Random Search terminé !")
print(f"\nMeilleurs hyperparamètres :")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nMeilleur score (CV) : {random_search.best_score_:.2%}")


### Modèle weighted avec meilleurs hyperparamètres

In [ ]:
# Modèle avec poids de classes ajustés
forest_weighted = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(
        n_estimators=300, random_state=42, max_depth=10, class_weight='balanced', min_samples_leaf=5, max_features='sqrt', min_samples_split=15
    ))
])


# Entraînement et détection d'overfitting
rf_y_pred_test_weighted, rf_y_pred_train_weighted = train_model_and_detect_overfitting(forest_weighted)


# Matrice de confusion et métriques - test
print('\n---RÉSULTATS DE LA RANDOM FOREST AVEC WEIGHTED SUR LE TEST\n')
display_metrics(y_test, rf_y_pred_test_weighted)


# Matrice de confusion et métriques - train
print('\n---RÉSULTATS DE LA RANDOM FOREST AVEC WEIGHTED SUR LE TRAIN\n')
display_metrics(y_train, rf_y_pred_train_weighted)

## Modèle avec Smote

### Recherche de la meilleure stratégie d'échantillonnage

In [ ]:
print(y_train.value_counts())

In [ ]:
forest_smote = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(random_state=42, n_estimators=300, max_depth=10, min_samples_leaf=5, max_features='sqrt', min_samples_split=15))
])


# Définition de la grille d'hyperparamètres
param_grid = {
    'smote__sampling_strategy': ['all', 'majority', {'gauche': 20000, 'centre': 23000}, {'gauche': 22000, 'centre': 25000}, {'gauche': 25000, 'centre': 25000}],
}


# Grid Search avec validation croisée
print("Lancement de Grid Search... (cela peut prendre quelques minutes)")
grid_search = GridSearchCV(
    forest_smote,
    param_grid,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=1
)


# Recherche des meilleurs hyperparamètres
grid_search.fit(X_train, y_train)

print("\n✓ Grid Search terminé !")
print(f"\nMeilleurs hyperparamètres :")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nMeilleur score (CV) : {grid_search.best_score_:.2%}")

### Recherche des meilleurs hyperparamètres - Random Search

In [ ]:
forest_smote = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42, sampling_strategy='all')),
    ('classifier', RandomForestClassifier(random_state=42))
])


param_distributions = {
    'classifier__n_estimators': [200, 250, 300],
    'classifier__max_depth': [5, 10, 15],
    'classifier__min_samples_split': [10, 12, 15],
    'classifier__min_samples_leaf': [1, 3, 5],
    'classifier__max_features': ['sqrt', 'log2']
}


# Random Search
print("Lancement de Random Search...")
random_search_smote = RandomizedSearchCV(
    forest_smote,
    param_distributions,
    n_iter=50,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    random_state=42,
    verbose=1
)


# Recherche
random_search_smote.fit(X_train, y_train)

print("\n✓ Random Search terminé !")
print(f"\nMeilleurs hyperparamètres :")
for param, value in random_search_smote.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nMeilleur score (CV) : {random_search_smote.best_score_:.2%}")


### Modèle Smote avec meilleurs hyperparamètres

In [ ]:
forest_smote = ImbPipeline([
    ('scaler', StandardScaler()),
    ('smote', SMOTE(random_state=42, sampling_strategy='all')),
    ('classifier', RandomForestClassifier(n_estimators=300, random_state=42, max_depth=10, min_samples_leaf=5, max_features='sqrt', min_samples_split=15))
])


# Entraînement
forest_smote.fit(X_train, y_train)

rf_y_pred_test_smote, rf_y_pred_train_smote = train_model_and_detect_overfitting(forest_smote)


# Métriques - test
print('\n---RÉSULTATS DE LA RANDOM FOREST AVEC SMOTE SUR LE TEST\n')
display_metrics(y_test, rf_y_pred_test_smote)


# Métriques - train
print('\n---RÉSULTATS DE LA RANDOM FOREST AVEC SMOTE SUR LE TRAIN\n')
display_metrics(y_train, rf_y_pred_train_smote)